# M3 historical-CLV gated TF-IDF neighbor residual — Dunnhumby seed 42

먼저 학습기간 내부의 마지막 5개 비중첩 7일 창에서 TF-IDF 유사사용자 관계가 일반 공동구매 전파와 degree-matched 무작위 관계보다 신규상품 후보를 더 잘 포함하는지 검사합니다. 사전 조건을 통과할 때만 M1·관계 대조군·실제 CLV·CLV shuffle·degree gate를 비교하는 100 epoch 학습을 실행합니다. final test와 holdout은 구성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, shutil, subprocess, sys

REVIEWED_SHA = 'a9c90dc4ac9b149ca26c18fe123a09432b2f09a5'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)

# 이전 Colab 실행에서 불러온 수정 전 모듈을 반드시 제거합니다.
for name in list(sys.modules):
    if name.startswith(('lightgcn_clv', 'clv_m3', 'clv_run_state')):
        del sys.modules[name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

In [ ]:
import importlib, json, torch
import lightgcn_clv_m3_tfidf_neighbor_diagnostic as diagnostic_runner
diagnostic_runner = importlib.reload(diagnostic_runner)
assert diagnostic_runner.CODE_VERSION == 'm3-clv-tfidf-neighbor-train-only-diagnostic-v1'
assert str(Path(diagnostic_runner.__file__).resolve()).startswith(str(repo.resolve()))

diagnostic_cfg = diagnostic_runner.configure_tfidf_neighbor_diagnostic()
assert diagnostic_cfg.train_end == 683
assert diagnostic_cfg.horizon_days == 7
assert diagnostic_cfg.n_anchors == 5
assert diagnostic_cfg.top_k_neighbors == 20
assert diagnostic_cfg.candidate_count == 100
print(json.dumps(diagnostic_cfg.__dict__, ensure_ascii=False, indent=2))

In [ ]:
from IPython.display import display

diagnostic_rows = diagnostic_runner.run_tfidf_neighbor_mechanism_diagnostic(diagnostic_cfg)
mechanism = diagnostic_rows.attrs['reading']
display(diagnostic_rows.groupby(['anchor_end', 'clv_group'])[[
    'tfidf_topk_neighbor',
    'ordinary_copurchase_propagation',
    'degree_matched_random_neighbor',
]].mean().reset_index())
print('\n학습 전 관계 판정:')
print(json.dumps(mechanism, ensure_ascii=False, indent=2))
print('\n진단 결과 파일:', diagnostic_rows.attrs['result_paths'])

In [ ]:
result_df = None
if not mechanism['precheck_passed']:
    print('사전 관계 진단이 통과되지 않아 4개 M3 arm의 고비용 학습을 실행하지 않습니다.')
else:
    assert torch.cuda.is_available(), 'Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.'
    import lightgcn_clv_m3_tfidf_neighbor_residual as m3_runner
    m3_runner = importlib.reload(m3_runner)
    assert m3_runner.CODE_VERSION == 'm3-clv-tfidf-neighbor-residual-historical-screen-v1'
    assert str(Path(m3_runner.__file__).resolve()).startswith(str(repo.resolve()))
    cfg = m3_runner.configure_tfidf_neighbor_residual_run()
    summary = m3_runner.preflight_summary(cfg)
    assert summary['seed'] == 42
    assert summary['historical_development_split']['final_test_constructed'] is False
    assert summary['historical_development_split']['holdout_constructed'] is False
    assert summary['m3']['historical_clv_proxy'] == 'N_hat * V_hat'
    assert summary['m3']['rho'] == 0.075
    assert summary['fixed']['binary_m1_graph_preserved'] is True
    assert summary['fixed']['negative_sampling'] == 'uniform'
    assert summary['fixed']['sample_weighting'] is False
    assert summary['fixed']['new_loss_term'] is False
    assert summary['fixed']['one_training_loop_and_optimizer'] is True
    assert summary['fixed']['pretraining_or_freezing'] is False
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    result_df = m3_runner.run_tfidf_neighbor_residual_screen(
        cfg, mechanism_reading=mechanism
    )

In [ ]:
if result_df is not None:
    columns = [
        'model_id', 'role',
        'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
        'recall@50', 'ndcg@50',
        'price_purchase_amount_weighted_hit@10',
        'mean_recommended_price_percentile@10',
        'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
        'eff_catalog@10', 'top10_share@10', 'top100_share@10',
        'eligible_user_share', 'eta_mean_eligible',
        'effective_budget_all', 'effective_budget_eligible',
        'max_absolute_orthogonality_error',
        'max_user_norm_absolute_error',
    ]
    display(result_df[[column for column in columns if column in result_df.columns]])
    print('\n실제 CLV 귀속 판정:')
    print(json.dumps(result_df.attrs['attribution_reading'], ensure_ascii=False, indent=2))
    print('\n결과 파일:', result_df.attrs['result_paths'])